# 綜合應用：精密軸承製造智慧製造案例

> **課程定位**：實務應用篇（6/6）  
> **前置課程**：[05_製程能力分析_Cp_Cpk](./05_製程能力分析_Cp_Cpk.ipynb)  
> **學習路徑**：01 概論 → 02 X-bar & R → 03 I-MR → 04 P Chart → 05 製程能力 → **06 綜合案例**

## 學習目標
- 整合所有 SPC 工具於一個完整製造場景
- 根據數據特性選擇適當的管制圖
- 執行「穩定性分析 → 能力分析 → 改善建議」完整流程
- 體驗智慧製造中數據驅動品質管理的實務應用

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Microsoft JhengHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

# ========== 工具函數（整合自前 5 堂課）==========

def plot_xbar_r(data_2d, title="X-bar & R 控制圖"):
    """繪製 X-bar & R 控制圖"""
    k, n = data_2d.shape
    xbar = data_2d.mean(axis=1)
    R = data_2d.max(axis=1) - data_2d.min(axis=1)
    xbar_bar = xbar.mean()
    R_bar = R.mean()
    
    A2_table = {2:1.880, 3:1.023, 4:0.729, 5:0.577, 6:0.483, 7:0.419, 8:0.373, 9:0.337, 10:0.308}
    D3_table = {2:0, 3:0, 4:0, 5:0, 6:0, 7:0.076, 8:0.136, 9:0.184, 10:0.223}
    D4_table = {2:3.267, 3:2.574, 4:2.282, 5:2.114, 6:2.004, 7:1.924, 8:1.864, 9:1.816, 10:1.777}
    
    UCL_x = xbar_bar + A2_table[n] * R_bar
    LCL_x = xbar_bar - A2_table[n] * R_bar
    UCL_r = D4_table[n] * R_bar
    LCL_r = D3_table[n] * R_bar
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    fig.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
    
    x_axis = range(1, k+1)
    ooc_x = (xbar > UCL_x) | (xbar < LCL_x)
    ax1.plot(x_axis, xbar, 'b-o', markersize=4, zorder=3)
    if ooc_x.any():
        ax1.scatter(np.where(ooc_x)[0]+1, xbar[ooc_x], color='red', s=80, zorder=5, marker='s')
    ax1.axhline(y=xbar_bar, color='green', linewidth=2, label=f'CL={xbar_bar:.4f}')
    ax1.axhline(y=UCL_x, color='red', linestyle='--', label=f'UCL={UCL_x:.4f}')
    ax1.axhline(y=LCL_x, color='red', linestyle='--', label=f'LCL={LCL_x:.4f}')
    ax1.set_ylabel('X-bar')
    ax1.set_title('X-bar 圖')
    ax1.legend(loc='upper right', fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    ooc_r = (R > UCL_r) | (R < LCL_r)
    ax2.plot(x_axis, R, 'r-s', markersize=4, zorder=3)
    if ooc_r.any():
        ax2.scatter(np.where(ooc_r)[0]+1, R[ooc_r], color='darkred', s=80, zorder=5, marker='D')
    ax2.axhline(y=R_bar, color='green', linewidth=2, label=f'CL={R_bar:.4f}')
    ax2.axhline(y=UCL_r, color='red', linestyle='--', label=f'UCL={UCL_r:.4f}')
    ax2.set_xlabel('子組編號')
    ax2.set_ylabel('R')
    ax2.set_title('R 圖')
    ax2.legend(loc='upper right', fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    return {'xbar':xbar, 'R':R, 'xbar_bar':xbar_bar, 'R_bar':R_bar,
            'UCL_x':UCL_x, 'LCL_x':LCL_x, 'UCL_r':UCL_r, 'LCL_r':LCL_r,
            'ooc_xbar':np.where(ooc_x)[0]+1, 'ooc_R':np.where(ooc_r)[0]+1}


def plot_imr(values, title="I-MR 控制圖"):
    """繪製 I-MR 控制圖"""
    values = np.array(values, dtype=float)
    n = len(values)
    MR = np.abs(np.diff(values))
    x_bar = values.mean()
    MR_bar = MR.mean()
    d2, D4 = 1.128, 3.267
    sigma_hat = MR_bar / d2
    UCL_I = x_bar + 3*sigma_hat
    LCL_I = x_bar - 3*sigma_hat
    UCL_MR = D4 * MR_bar
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    fig.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
    
    ooc_I = (values > UCL_I) | (values < LCL_I)
    ax1.plot(range(1,n+1), values, 'b-o', markersize=4, zorder=3)
    if ooc_I.any():
        ax1.scatter(np.where(ooc_I)[0]+1, values[ooc_I], color='red', s=80, zorder=5, marker='s')
    ax1.axhline(y=x_bar, color='green', linewidth=2, label=f'CL={x_bar:.4f}')
    ax1.axhline(y=UCL_I, color='red', linestyle='--', label=f'UCL={UCL_I:.4f}')
    ax1.axhline(y=LCL_I, color='red', linestyle='--', label=f'LCL={LCL_I:.4f}')
    ax1.set_ylabel('個別值')
    ax1.set_title('I 圖')
    ax1.legend(loc='upper right', fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    ooc_MR = MR > UCL_MR
    ax2.plot(range(2,n+1), MR, 'r-s', markersize=4, zorder=3)
    if ooc_MR.any():
        ax2.scatter(np.where(ooc_MR)[0]+2, MR[ooc_MR], color='darkred', s=80, zorder=5, marker='D')
    ax2.axhline(y=MR_bar, color='green', linewidth=2, label=f'CL={MR_bar:.4f}')
    ax2.axhline(y=UCL_MR, color='red', linestyle='--', label=f'UCL={UCL_MR:.4f}')
    ax2.set_xlabel('樣本編號')
    ax2.set_ylabel('移動全距')
    ax2.set_title('MR 圖')
    ax2.legend(loc='upper right', fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    return {'x_bar':x_bar, 'MR_bar':MR_bar, 'UCL_I':UCL_I, 'LCL_I':LCL_I,
            'ooc_I':np.where(ooc_I)[0]+1, 'ooc_MR':np.where(ooc_MR)[0]+2}


def plot_p_chart(defects, sample_sizes, title="P 管制圖"):
    """繪製 P 管制圖"""
    defects = np.array(defects, dtype=float)
    sample_sizes = np.array(sample_sizes, dtype=float)
    k = len(defects)
    p = defects / sample_sizes
    p_bar = defects.sum() / sample_sizes.sum()
    UCL = p_bar + 3*np.sqrt(p_bar*(1-p_bar)/sample_sizes)
    LCL = np.maximum(0, p_bar - 3*np.sqrt(p_bar*(1-p_bar)/sample_sizes))
    
    fig, ax = plt.subplots(figsize=(14, 7))
    ooc = (p > UCL) | (p < LCL)
    ax.plot(range(1,k+1), p, 'b-o', markersize=5, zorder=3)
    if ooc.any():
        ax.scatter(np.where(ooc)[0]+1, p[ooc], color='red', s=100, zorder=5, marker='s')
    ax.axhline(y=p_bar, color='green', linewidth=2, label=f'p\u0304={p_bar:.4f}')
    if np.all(sample_sizes == sample_sizes[0]):
        ax.axhline(y=UCL[0], color='red', linestyle='--', label=f'UCL={UCL[0]:.4f}')
        ax.axhline(y=LCL[0], color='red', linestyle='--', label=f'LCL={LCL[0]:.4f}')
    else:
        ax.step(range(1,k+1), UCL, color='red', linestyle='--', where='mid', label='UCL')
        ax.step(range(1,k+1), LCL, color='red', linestyle='--', where='mid', label='LCL')
    ax.set_title(title, fontsize=16, fontweight='bold')
    ax.set_xlabel('子組編號')
    ax.set_ylabel('不良率')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return {'p':p, 'p_bar':p_bar, 'UCL':UCL, 'LCL':LCL, 'ooc':np.where(ooc)[0]+1}


def process_capability_report(data, USL, LSL, target=None, title="製程能力分析"):
    """生成製程能力分析報告"""
    data = np.array(data, dtype=float)
    mean = data.mean()
    std = data.std(ddof=1)
    if target is None:
        target = (USL + LSL) / 2
    
    Cp = (USL - LSL) / (6*std)
    Cpu = (USL - mean) / (3*std)
    Cpl = (mean - LSL) / (3*std)
    Cpk = min(Cpu, Cpl)
    ppm_upper = stats.norm.sf((USL-mean)/std) * 1e6
    ppm_lower = stats.norm.cdf((LSL-mean)/std) * 1e6
    ppm_total = ppm_upper + ppm_lower
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(data, bins=20, density=True, alpha=0.7, color='steelblue', edgecolor='white')
    x_fit = np.linspace(data.min()-std, data.max()+std, 300)
    ax.plot(x_fit, stats.norm.pdf(x_fit, mean, std), 'b-', linewidth=2)
    ax.axvline(x=USL, color='red', linestyle='--', linewidth=2, label=f'USL={USL}')
    ax.axvline(x=LSL, color='red', linestyle='--', linewidth=2, label=f'LSL={LSL}')
    ax.axvline(x=mean, color='orange', linewidth=2, label=f'Mean={mean:.3f}')
    ax.set_title(f'{title}\nCp={Cp:.3f}, Cpk={Cpk:.3f}, PPM={ppm_total:.1f}', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return {'Cp':Cp, 'Cpk':Cpk, 'Cpu':Cpu, 'Cpl':Cpl, 'ppm_total':ppm_total, 'mean':mean, 'std':std}

print("\u2705 所有 SPC 工具函數已載入")

## 1. 案例背景：精密軸承製造產線

某精密軸承製造商生產高精度深溝球軸承，監控三項品質特性：

| 品質特性 | 量測方式 | 數據類型 | 適用管制圖 | 規格 |
|----------|----------|----------|------------|------|
| 內環直徑 | 每班取 5 件量測 | 連續型，子組 | **X-bar & R** | 25.000 \u00b1 0.015 mm |
| 表面粗糙度 Ra | 每班 1 次破壞性測試 | 連續型，個別值 | **I-MR** | Ra \u2264 0.20 \u03bcm |
| 外觀不良率 | 每班全檢 ~300 件 | 計數型 | **P Chart** | \u2014 |

### 數據收集
- 收集 **50 個班次**（約 17 天三班制）的數據
- 已知問題：
  - 班次 35 後：刀具磨損導致內環直徑**趨勢漂移**
  - 班次 22：研磨液汙染導致表面粗糙度**突然偏高**
  - 班次 40：新進人員操作不當導致外觀不良率**突升**

In [ ]:
np.random.seed(2024)
n_shifts = 50

# ===== 品質特性 1: 內環直徑 (X-bar & R, n=5) =====
# 前 34 班穩定，35 班後刀具磨損導致逐漸漂移
diameter_data = np.zeros((n_shifts, 5))
for i in range(n_shifts):
    if i < 34:
        diameter_data[i] = np.random.normal(loc=25.000, scale=0.003, size=5)
    else:
        drift = 0.001 * (i - 33)  # 每班漂移 0.001mm
        diameter_data[i] = np.random.normal(loc=25.000 + drift, scale=0.003, size=5)

# ===== 品質特性 2: 表面粗糙度 Ra (I-MR, 個別值) =====
roughness_data = np.random.normal(loc=0.12, scale=0.02, size=n_shifts)
roughness_data[21] = 0.25  # 班次 22 研磨液汙染

# ===== 品質特性 3: 外觀不良率 (P Chart) =====
inspection_counts = np.random.choice([280, 300, 320], size=n_shifts)
defect_counts = np.array([
    np.random.binomial(n=inspection_counts[i], p=0.02) for i in range(n_shifts)
])
defect_counts[39] = np.random.binomial(n=inspection_counts[39], p=0.10)  # 班次 40 新人問題

print("=" * 60)
print("精密軸承製造產線 - 數據總覽")
print("=" * 60)

# 建立總覽 DataFrame
df_overview = pd.DataFrame({
    '班次': range(1, n_shifts+1),
    '內環直徑_平均 (mm)': diameter_data.mean(axis=1).round(4),
    '內環直徑_全距 (mm)': (diameter_data.max(axis=1) - diameter_data.min(axis=1)).round(4),
    '表面粗糙度 Ra (\u03bcm)': roughness_data.round(4),
    '檢驗數量': inspection_counts,
    '外觀不良品數': defect_counts,
    '外觀不良率': (defect_counts / inspection_counts).round(4)
})

print("\n前 10 個班次：")
display(df_overview.head(10))
print(f"\n總班次數: {n_shifts}")
print(f"內環直徑 - 總平均: {diameter_data.mean():.4f} mm")
print(f"表面粗糙度 - 平均: {roughness_data.mean():.4f} \u03bcm")
print(f"外觀不良率 - 平均: {defect_counts.sum()/inspection_counts.sum():.2%}")

## 2. Phase 1：穩定性分析（管制圖）

### 2.1 內環直徑：X-bar & R 圖

In [ ]:
print("=" * 60)
print("品質特性 1: 內環直徑 X-bar & R 分析")
print("=" * 60)
result_diameter = plot_xbar_r(diameter_data, title="內環直徑 X-bar & R 控制圖")
print(f"\nX-bar 圖超出管制: 班次 {list(result_diameter['ooc_xbar'])}")
print(f"R 圖超出管制:     班次 {list(result_diameter['ooc_R'])}")
print(f"\n分析：第 35 班後 X-bar 持續上升 \u2192 刀具磨損導致的趨勢性漂移")
print("R 圖相對穩定 \u2192 變異性未增加，問題在於平均值偏移")

### 2.2 表面粗糙度 Ra：I-MR 圖

In [ ]:
print("=" * 60)
print("品質特性 2: 表面粗糙度 Ra I-MR 分析")
print("=" * 60)
result_roughness = plot_imr(roughness_data, title="表面粗糙度 Ra I-MR 控制圖")
print(f"\nI 圖超出管制:  樣本 {list(result_roughness['ooc_I'])}")
print(f"MR 圖超出管制: 樣本 {list(result_roughness['ooc_MR'])}")
print(f"\n分析：班次 22 明顯偏高（研磨液汙染事件）")
print("MR 圖在班次 22-23 出現大跳動 \u2192 確認是突發事件而非趨勢")

### 2.3 外觀不良率：P Chart

In [ ]:
print("=" * 60)
print("品質特性 3: 外觀不良率 P 圖分析")
print("=" * 60)
result_defect = plot_p_chart(defect_counts, inspection_counts, 
                              title="外觀不良率 P 管制圖")
print(f"\n超出管制: 班次 {list(result_defect['ooc'])}")
print(f"\n分析：班次 40 不良率突升 \u2192 新進人員操作不當")
print("其餘班次穩定 \u2192 為偶發事件，非系統性問題")

## 3. Phase 2：製程能力分析

排除特殊原因後，對穩定區段進行能力分析。

### 3.1 內環直徑能力分析
- 規格：25.000 \u00b1 0.015 mm（USL=25.015, LSL=24.985）
- 使用前 34 班（穩定段）的數據

In [ ]:
# 使用穩定段數據
stable_diameter = diameter_data[:34].flatten()
report_dia = process_capability_report(stable_diameter, USL=25.015, LSL=24.985, target=25.000,
                                        title="內環直徑製程能力分析（穩定段）")

print(f"\n內環直徑能力評估：")
print(f"  Cp  = {report_dia['Cp']:.3f}")
print(f"  Cpk = {report_dia['Cpk']:.3f}")
print(f"  PPM = {report_dia['ppm_total']:.1f}")
if report_dia['Cpk'] >= 1.33:
    print("  \u2705 製程能力合格")
else:
    print("  \u26a0\ufe0f 製程能力不足，需要改善")

### 3.2 表面粗糙度能力分析
- 規格：Ra \u2264 0.20 \u03bcm（僅有上限，使用單側 Cpk）
- 排除班次 22 的異常數據

In [ ]:
# 排除班次 22
stable_roughness = np.delete(roughness_data, 21)

mean_r = stable_roughness.mean()
std_r = stable_roughness.std(ddof=1)
USL_r = 0.20

# 單側 Cpk
Cpu_r = (USL_r - mean_r) / (3 * std_r)
ppm_r = stats.norm.sf((USL_r - mean_r) / std_r) * 1e6

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(stable_roughness, bins=15, density=True, alpha=0.7, color='steelblue', edgecolor='white')
x_fit = np.linspace(stable_roughness.min()-0.02, 0.25, 200)
ax.plot(x_fit, stats.norm.pdf(x_fit, mean_r, std_r), 'b-', linewidth=2)
ax.axvline(x=USL_r, color='red', linestyle='--', linewidth=2, label=f'USL = {USL_r} \u03bcm')
ax.axvline(x=mean_r, color='orange', linewidth=2, label=f'Mean = {mean_r:.4f} \u03bcm')
ax.set_title(f'表面粗糙度 Ra 製程能力分析\nCpu = {Cpu_r:.3f}, PPM = {ppm_r:.1f}', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Ra (\u03bcm)')
ax.set_ylabel('機率密度')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"表面粗糙度能力評估（單側規格）：")
print(f"  平均值: {mean_r:.4f} \u03bcm")
print(f"  標準差: {std_r:.4f} \u03bcm")
print(f"  Cpu = {Cpu_r:.3f}")
print(f"  預估超出 USL 的 PPM: {ppm_r:.1f}")

## 4. Phase 3：改善建議與持續監控

### 異常事件摘要

| 品質特性 | 異常班次 | 原因 | 改善行動 |
|----------|----------|------|----------|
| 內環直徑 | 35~50 | 刀具磨損趨勢漂移 | 建立刀具壽命管理，定期換刀 |
| 表面粗糙度 | 22 | 研磨液汙染 | 加強研磨液更換 SOP |
| 外觀不良率 | 40 | 新人操作不當 | 強化新人訓練與考核 |

### 修正管制界限

排除特殊原因後，重新計算管制界限作為未來監控的基準。

In [ ]:
print("=" * 60)
print("修正後管制界限（排除特殊原因）")
print("=" * 60)

# 內環直徑：使用前 34 班
print("\n\U0001f4cf 內環直徑（前 34 班穩定段）：")
result_revised = plot_xbar_r(diameter_data[:34], title="內環直徑 修正後 X-bar & R 控制圖")
print(f"  X-bar: CL={result_revised['xbar_bar']:.4f}, UCL={result_revised['UCL_x']:.4f}, LCL={result_revised['LCL_x']:.4f}")

# 表面粗糙度：排除班次 22
print("\n\U0001f4cf 表面粗糙度（排除班次 22）：")
result_revised_r = plot_imr(stable_roughness, title="表面粗糙度 修正後 I-MR 控制圖")

## 5. SPC 工具箱：管制圖選擇決策指南

```
數據類型是什麼？
\u251c\u2500\u2500 連續型（量測值）
\u2502   \u251c\u2500\u2500 有子組（每次多個樣本）？
\u2502   \u2502   \u251c\u2500\u2500 n = 2~10 \u2192 X-bar & R 圖
\u2502   \u2502   \u2514\u2500\u2500 n > 10 \u2192 X-bar & S 圖
\u2502   \u2514\u2500\u2500 個別值（每次一個樣本）
\u2502       \u2514\u2500\u2500 I-MR 圖
\u2502           \u26a0\ufe0f 需先做常態性檢驗
\u2514\u2500\u2500 離散型（計數）
    \u251c\u2500\u2500 不良品比例/數量？
    \u2502   \u251c\u2500\u2500 樣本大小固定 \u2192 np 圖
    \u2502   \u2514\u2500\u2500 樣本大小可變 \u2192 P 圖
    \u2514\u2500\u2500 缺陷數/密度？
        \u251c\u2500\u2500 檢驗範圍固定 \u2192 C 圖
        \u2514\u2500\u2500 檢驗範圍可變 \u2192 u 圖
```

### 完整 SPC 分析流程

1. **收集數據** \u2192 選擇適當的管制圖
2. **穩定性分析** \u2192 繪製控制圖，識別異常
3. **消除特殊原因** \u2192 調查並排除異常
4. **重新計算管制界限** \u2192 修正後的基準
5. **能力分析** \u2192 計算 Cp/Cpk
6. **持續監控** \u2192 使用修正後的管制界限

## 6. 本章小結

| 階段 | 內容 | 工具 |
|------|------|------|
| Phase 1 | 穩定性分析 | X-bar & R、I-MR、P Chart |
| Phase 2 | 能力分析 | Cp、Cpk（排除異常後） |
| Phase 3 | 改善建議 | 修正管制界限 + 行動計畫 |

### 本系列課程回顧

| 課程 | 核心內容 | 關鍵函數 |
|------|----------|----------|
| 01 概論 | SPC 定義、變異類型、3-Sigma | \u2014 |
| 02 X-bar & R | 子組型連續數據管制圖 | `plot_xbar_r()` |
| 03 I-MR | 個別值連續數據管制圖 | `plot_imr()` |
| 04 P Chart | 計數型數據管制圖 | `plot_p_chart()` |
| 05 Cp/Cpk | 製程能力分析 | `process_capability_report()` |
| 06 綜合案例 | 完整 SPC 流程整合 | 所有工具 |

---

## 課堂練習

### \U0001f7e2 基礎題
1. 根據以上案例，哪個品質特性的問題最嚴重？為什麼？

### \U0001f7e1 進階題
2. 假設修復刀具磨損問題後，又收集了 20 班的新數據。用以下模擬數據繪製控制圖，確認製程是否恢復穩定：
```python
np.random.seed(999)
new_diameter = np.random.normal(loc=25.000, scale=0.003, size=(20, 5))
```

### \U0001f534 挑戰題
3. 設計一個「SPC 自動化監控儀表板」：
   - 輸入：新的量測數據
   - 輸出：自動判斷管制狀態、標記異常、計算能力指數
   - 提示：將所有工具函數封裝進一個 `SPCDashboard` 類別

---

## 延伸閱讀

- **進階管制圖**：CUSUM（累積和）圖、EWMA（指數加權移動平均）圖
- **多變量 SPC**：T\u00b2 控制圖、MEWMA
- **智慧製造整合**：SPC + 機器學習、即時異常偵測
- **推薦書籍**：Montgomery,《Introduction to Statistical Quality Control》第 8 版

> \U0001f393 **恭喜完成 SPC 系列課程！** 你現在具備了運用統計製程管制工具進行品質管理的核心能力。